# Multi-year NDVI change — the OPTIONAL geemap / Earth Engine path

This mirrors `01_stac_change.ipynb` on **Google Earth Engine** for parity with [`giswqs/geemap`](https://github.com/giswqs/geemap). It is **not required** to run the project: the auth-free STAC notebook is the default. Use this only if you have an Earth Engine account.

**Setup (one-time, free Google account):**

```bash
pip install earthengine-api geemap
earthengine authenticate   # opens a browser, free Google sign-in
```

In [ ]:
import ee
import geemap
import yaml

# Replace with your Earth Engine Cloud project id (see .env.example).
ee.Initialize(project="your-earth-engine-project-id")

In [ ]:
with open("../config/aoi.yaml") as fh:
    cfg = yaml.safe_load(fh)
b = cfg["aoi"]["bbox"]
bbox = [b["min_lon"], b["min_lat"], b["max_lon"], b["max_lat"]]
a = cfg["analysis"]
baseline = (cfg["baseline_years"]["start"], cfg["baseline_years"]["end"])
recent = (cfg["recent_years"]["start"], cfg["recent_years"]["end"])

## Server-side composites and change

`eets.gee.change_composite` builds the cloud-masked median NDVI for each period on the Earth Engine servers and returns `before`, `after`, and `change = after - before` as `ee.Image` objects — the exact analogue of the STAC `build_change_inputs`.

In [ ]:
from eets.gee import change_composite

out = change_composite(
    bbox,
    baseline_years=baseline,
    recent_years=recent,
    index=a["index"],
    max_cloud=a["max_cloud"],
)
before_img, after_img, change_img = out["before"], out["after"], out["change"]

## Map the change

Threshold the change image to highlight NDVI loss.

In [ ]:
region = ee.Geometry.Rectangle(bbox)
loss = change_img.lte(a["loss_thresh"])

m = geemap.Map()
m.centerObject(region, 12)
vis = {"min": 0, "max": 1, "palette": ["white", "green"]}
m.addLayer(before_img, vis, "Baseline NDVI")
m.addLayer(after_img, vis, "Recent NDVI")
m.addLayer(loss.selfMask(), {"palette": ["red"]}, "NDVI loss")
m

## Loss area in hectares (server-side)

Sum the loss pixels' area on the server and convert to hectares — the Earth Engine equivalent of `change_stats`. Compare this back to the STAC notebook's number and to Global Forest Watch / Hansen for the AOI.

In [ ]:
area_img = loss.multiply(ee.Image.pixelArea())  # m^2 per loss pixel
total_m2 = area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=a["pixel_size_m"],
    maxPixels=1e10,
).get(change_img.bandNames().get(0))
print("Earth Engine NDVI-loss area (ha):", ee.Number(total_m2).divide(1e4).getInfo())

Same logic, two engines: the STAC path needs no account and is the tested default; Earth Engine is here for parity and convenience if you already use it.